# RAG

# Set up

In [1]:
%pip install pymilvus==2.6.9 openai==2.20.0

^C
Note: you may need to restart the kernel to use updated packages.


     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     --------------------------- ------------ 61.4/90.6 kB 3.4 MB/s eta 0:00:01
     ---------------------------------------- 90.6/90.6 kB 1.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ----------------------------------- ---- 51.2/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/301.2 kB ? eta -:--:--
   ------------------- -------------------- 143.4/301.2 kB 4.2 MB/s eta 0:00:01
   --------------------------------- ------ 256.0/301.2 kB 3.1 MB/s eta 0:00:01
   ---------------------------------------  297.0/301.2 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 301.2/301.2 kB 1.5 MB/s eta 0:00:00


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'k:\\Gen AI\\project\\rag_4_retail\\retail_rag_venv\\Lib\\site-packages\\pydantic\\generics.py'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
%pip install milvus-lite==2.5.1

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement milvus-lite==2.5.1 (from versions: none)
ERROR: No matching distribution found for milvus-lite==2.5.1


In [1]:
%pwd

'k:\\Gen AI\\project\\rag_4_retail\\notebook'

In [2]:
%cd ..

k:\Gen AI\project\rag_4_retail


In [4]:
!powershell -Command "pip list | Select-String 'milvus_lite'"

In [5]:
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from env directory
env_path = Path("./env/.env")
load_dotenv(env_path)

print(f"Loaded env file from: {env_path.resolve()}")

Loaded env file from: K:\Gen AI\project\rag_4_retail\env\.env


# 1) Preparation 

## Prepare the data

In [6]:
from glob import glob

text_lines = []

for file_path in glob("./data/milvus_docs_2.4.x_en/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")


In [7]:
len(text_lines)

72

In [8]:
text_lines[0: 2]

['---\nid: operational_faq.md\nsummary: Find answers to commonly asked questions about operations in Milvus.\ntitle: Operational FAQ\n---\n\n',
 'Operational FAQ\n\n<!-- TOC -->\n\n\n<!-- /TOC -->\n\n###']

## Prepare the Embedding Model 

In [9]:
from openai import OpenAI

openai_client = OpenAI()

openai_client

In [ ]:
def embed_text(text):
    embedding_res = openai_client.embeddings.create(input=text, model="text-embedding-3-small")
    embedding = embedding_res.data[0].embedding
    
    return {
        'embedding_res': embedding_res, 
        'embedding': embedding
    }


In [ ]:
test_embedding_dict = embed_text("This is a test")

test_embedding = test_embedding_dict.get('embedding')
test_embedding_dim = len(test_embedding)

print('test_embedding_dim =', test_embedding_dim,' \n')
print('test_embedding[: 10] =', test_embedding[: 10])


test_embedding_dim = 1536  

test_embedding[: 10] = [0.009915309958159924, -0.0055878316052258015, 0.006812288425862789, -0.03809776529669762, -0.018203329294919968, -0.04122471809387207, -0.007665819022804499, 0.03221079707145691, 0.018937205895781517, 9.491286618867889e-05]


# 2) Load data into Milvus

## Create the Collection 

In [14]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="http://localhost:19530")

collection_name = "my_rag_collection"

In [17]:
milvus_client

In [18]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

    print(collection_name, 'exist')

In [19]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=test_embedding_dim,
    metric_type="IP",  # Inner product distance
    consistency_level="Bounded",  # Supported values are (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`). See https://milvus.io/docs/consistency.md#Consistency-Level for more details.
)


## Insert data 

In [20]:
milvus_client.has_collection(collection_name)

True

In [22]:
from tqdm import tqdm

data = []

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):

    embedding = embed_text(line).get('embedding')
    
    data.append({"id": i, "vector": embedding, "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 72/72 [00:22<00:00,  3.16it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

# 3) Build RAG 

## Retrieve data for a query 

In [23]:
question = "How is data stored in milvus?"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=[
        embed_text(question).get('embedding')
    ],  # Use the `emb_text` function to convert the question to an embedding vector
    limit=3,  # Return top 3 results
    search_params={"metric_type": "IP", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
)

In [30]:
type(search_res)

pymilvus.client.search_result.SearchResult

In [24]:
search_res

data: [[{'id': 31, 'distance': 0.7827308177947998, 'entity': {'text': ' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###'}}, {'id': 57, 'distance': 0.6772557497024536, 'entity': {'text': 'How does Milvus ha

In [28]:
# Results is a list of lists
for hits in search_res:
    for hit in hits:
        print(hit.id)        # Document ID
        print(hit.score)     # Similarity score
        print(hit.entity)    # Entity data including "text" field
        print()

31
0.7827308177947998
{'id': 31, 'distance': 0.7827308177947998, 'entity': {'text': ' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###'}}

57
0.6772557497024536
{'id': 57, 'distance': 0.6772557497024536, 'e

In [31]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))


[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.7827308177947998
    ],
    [
        "How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float

## Use LLM to get a RAG response 

Convert the retrieved documents into a string format.

In [32]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

context

" Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float32, Float16, and BFloat16 vector types.\n\n- Binary vectors: Store

Define system and user prompts for the Language Model. This prompt is assembled with the retrieved documents from Milvus.

In [33]:
SYSTEM_PROMPT = """
Human: You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""

USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
<context>
{context}
</context>
<question>
{question}
</question>
"""


In [41]:
response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)

response

ChatCompletion(id='chatcmpl-D9vWUSPS7an6qCTUpgHnJC5z6RpeZ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Milvus stores data in two ways: inserted data, which includes vector data, scalar data, and collection-specific schema, is stored in persistent storage as an incremental log. Multiple object storage backends are supported for this purpose such as MinIO, AWS S3, Google Cloud Storage, Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage. Metadata generated within Milvus, on the other hand, is stored in etcd.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1771258442, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=90, prompt_tokens=546, total_tokens=636, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasonin

In [42]:
dir(response)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__

In [43]:
print(response.choices[0].message.content)

Milvus stores data in two ways: inserted data, which includes vector data, scalar data, and collection-specific schema, is stored in persistent storage as an incremental log. Multiple object storage backends are supported for this purpose such as MinIO, AWS S3, Google Cloud Storage, Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage. Metadata generated within Milvus, on the other hand, is stored in etcd.
